# Text Classification #

### 20newsgroups dataset ###
20,000 newsgroup documents, partitioned (nearly) evenly across 20 different newsgroups.

In [21]:
from sklearn.datasets import load_files

TRAIN_DIR = "20news-bydate-train"

train = load_files(
    TRAIN_DIR,
    encoding="latin1",
    shuffle=True,
    random_state=42
)


In [22]:
print('Train set size: %s ' % train.target.size)

Train set size: 11314 


In [23]:
print('FIRST TEXT CATEGORY: %s \n\n' % train.target_names[train.target[0]])
print('FIRST TEXT: \n')
print('\n'.join(train.data[0].split("\n")[:10])) 

FIRST TEXT CATEGORY: sci.electronics 


FIRST TEXT: 

From: wtm@uhura.neoucom.edu (Bill Mayhew)
Subject: Re: How to the disks copy protected.
Organization: Northeastern Ohio Universities College of Medicine
Lines: 23

Write a good manual to go with the software.  The hassle of
photocopying the manual is offset by simplicity of purchasing
the package for only $15.  Also, consider offering an inexpensive
but attractive perc for registered users.  For instance, a coffee
mug.  You could produce and mail the incentive for a couple of


# 1. Bag of Words  - data representation #

### Vectorization ###

In [24]:
import numpy as np
np.set_printoptions(precision=2)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

In [25]:
corpus = [
    'aaa aaa aaa aaa aaa bbb',
    'bbb bbb bbb bbb bbb bbb',
    'bbb ccc',
   ]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

print(vectorizer.get_feature_names_out())
X.toarray()

['aaa' 'bbb' 'ccc']


array([[5, 1, 0],
       [0, 6, 0],
       [0, 1, 1]])

### TF-IDF (TF â term frequency, IDF â inverse document frequency) ###

In [26]:
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X)
X_train_tfidf.toarray()

array([[0.99, 0.12, 0.  ],
       [0.  , 1.  , 0.  ],
       [0.  , 0.51, 0.86]])

# 2. Fitting a model, Pipeline #

In [27]:
# Vectorization
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(train.data)
X_train_counts.shape

(11314, 130107)

In [28]:
# Converting to TF-IDF
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_train_tfidf.shape

(11314, 130107)

In [29]:
# Using DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
# dtc = DecisionTreeClassifier().fit(X_train_tfidf, train.target)


### Pipeline ###

In [30]:
# We can write less code and do all of the above, by building a pipeline.
# The names âvectâ , âtfidfâ and âclfâ are arbitrary.
# The purpose of the pipeline is to assemble several steps that can be
# cross-validated together while setting different parameters.

from sklearn.pipeline import Pipeline

pipe_clf = Pipeline([
    ('vect', CountVectorizer()), 
    ('tfidf', TfidfTransformer()), 
    ('dtc', DecisionTreeClassifier())
])

# Now we can use orginal dataset train.data
pipe_clf = pipe_clf.fit(train.data, train.target)

In [31]:
# Performance of DecisionTreeClassifier
TEST_DIR = "20news-bydate-test"

test = load_files(
    TEST_DIR,
    encoding="latin1",   # WAŻNE: oryginalny encoding datasetu
    shuffle=True,
    random_state=42
)

predicted = pipe_clf.predict(test.data)
np.mean(predicted == test.target)

# is the result realy bad?

np.float64(0.5535050451407328)

### Grid search ###

In [32]:
# Create a list of parameters and their values to be checked.
# All the parameters name are of the form 'stepName__paramName'.
# E.g. 'vect__ngram_range': [(1, 1), (1, 2)]
# that means use unigram and bigrams and choose the one which is optimal.

parameters = {
    'vect__ngram_range': [(1, 1),(1, 2)],  
    'tfidf__use_idf': (True, False)
#     'dtc__max_depth': (20,40)
}

In [18]:
#BELOW COMMANDS ARE TIME EXPENSIVE!

# n_jobs=-1 means using all cores
# Perheps you may need to run "conda install -c anaconda joblib" 

from sklearn.model_selection import GridSearchCV

gs_clf = GridSearchCV(pipe_clf, parameters, n_jobs=-1)

# Run the grid search on the pipeline
gs_clf = gs_clf.fit(train.data, train.target)
print("Best score: %s" % gs_clf.best_score_) 
print("Best param: %s" % gs_clf.best_params_) 

Best score: 0.6415949032859756
Best param: {'tfidf__use_idf': False, 'vect__ngram_range': (1, 2)}


# 3. NLTK - Natural Language Toolkit #

### Stop words ###

In [33]:
# # Removing stop words with CountVectorizer
text_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english')), 
    ('tfidf', TfidfTransformer()), 
    ('clf', DecisionTreeClassifier())
])

In [34]:
#!pip install nltk
import nltk
nltk.download('snowball_data')
nltk.download('stopwords')

from nltk.corpus import stopwords
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package snowball_data to
[nltk_data]     C:\Users\woks0\AppData\Roaming\nltk_data...
[nltk_data]   Package snowball_data is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\woks0\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Stemming ###

In [35]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("english", ignore_stopwords=True)
print('running --> %s' % stemmer.stem("running"))
print('generously --> %s' %stemmer.stem("generously"))



running --> run
generously --> generous


In [36]:
# Use stemming in the vectorization process

class StemmedCountVectorizer(CountVectorizer):
    def build_analyzer(self):
        analyzer = super(StemmedCountVectorizer, self).build_analyzer()
        return lambda doc: ([stemmer.stem(w) for w in analyzer(doc)])
    
stemmed_count_vect = StemmedCountVectorizer(stop_words='english')

pipe_stemmed = Pipeline([
    ('vect', stemmed_count_vect),
    ('tfidf', TfidfTransformer()), 
    ('dtc', DecisionTreeClassifier())
])

pipe_stemmed = pipe_stemmed.fit(train.data, train.target)

predicted_stemmed = pipe_stemmed.predict(test.data)

from sklearn.linear_model import LogisticRegression
print('Accuracy after stemming: %s' % np.mean(predicted_stemmed == test.target))

Accuracy after stemming: 0.5723579394583113


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.model_selection import cross_val_score

parameters = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'vect__min_df': [3, 5],
    'vect__max_df': [0.7, 0.9],
    
    'tfidf__use_idf': [True, False],
    
    'clf__C': [0.1, 1, 10],
}

pipe_stemmed = Pipeline([
    ('vect', stemmed_count_vect),
    ('tfidf', TfidfTransformer()), 
    ('clf', LogisticRegression(max_iter=1000))
])

from sklearn.model_selection import GridSearchCV

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = GridSearchCV(
    estimator=pipe_stemmed,
    param_grid=parameters,
    cv=kfold,
    n_jobs=-1,
    scoring='accuracy'
)

model.fit(train.data, train.target)

best_model = model.best_estimator_

print(model.best_params_)


#walidacja krzyzowa na zbiorze testowym
scores = cross_val_score(best_model, test.data, test.target, cv=kfold, n_jobs=-1)
print("Cross validation scores: ", scores)
print("Mean cross validation score: %.4f" % scores.mean())




{'clf__C': 10, 'tfidf__use_idf': True, 'vect__max_df': 0.7, 'vect__min_df': 3, 'vect__ngram_range': (1, 2)}
Cross validation scores:  [0.9  0.89 0.9  0.91 0.92]
Mean cross validation score: 0.9041
